## 의사결정나무
- 질문에 따라 판단
- 젤 위 루트 노드 --> 분할 --> 분할 ...
- 깊이가 깊어질때 과적합이 발생할 위험이 크다.
- 분류/회귀 둘 다 가능하다.

In [ ]:
# 실습 폴더 불러오기
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('ml_data')
if not DATA_DIR.exists():
    DATA_DIR = Path('ml_data')
print(f'데이터 폴더: {DATA_DIR.resolve()}')

데이터 폴더: C:\Users\Administrator\bigdata2026\data_analysis\machine_learning\ml_실습데이터


### 라이브러리 불러오기

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
# export_text : 나무 구조를 글자로 출력
from sklearn.tree import DecisionTreeClassifier, export_text # 의사결정나무 분류
from sklearn.impute import SimpleImputer  # 결측치를 특정 규칙에 따라 채워주는 전처리 도구
from sklearn.pipeline import make_pipeline # 여러 전처리 단계와 모델을 하나의 객체로 순서대로 연결

### 데이터 불러오기

In [22]:
df=pd.read_csv(DATA_DIR/'telecom_churn.csv')
df.head()

,usage_minutes,complaints,contract_months,monthly_fee,contract_type,region,churn
0,321.0,2,12,63.6,one-year,Gyeonggi,1
1,80.0,1,6,62.2,two-year,Gyeonggi,1
2,251.0,1,31,71.6,month-to-month,Other,0
3,158.0,0,31,58.8,two-year,Seoul,0
4,356.0,1,32,58.9,one-year,Gyeonggi,0


### 피처(입력) / 타겟(정답) 데이터 나누기

In [23]:
# usage_minutes : 고객의 (월간) 통화/이용 시간(분) -> 이용량이 급격히 줄어든 고객은 이미 다른 서비스로 떠났다.
# complaints : 고객이 접수한 불만/민원 건수 --> 서비스에 불만족해서 이탈할 확률이 높다고 판단
# contract_months: 계약 유지 기간(개월 수) --> 계약 기간이 짧을 수록 이탈 위험이 큰 것으로 판단
# monthly_fee : 월 요금 (청구액) --> 요금이 부담스러울수록 이탈 가능성이 크다.
# churn : 타겟(정답), 이탈한다(1)/안한다(0)
features = ['usage_minutes', 'complaints', 'contract_months', 'monthly_fee']
X = df[features]
y = df['churn']

X.shape, y.shape

((420, 4), (420,))

### 훈련 / 검증 데이터 나누기

In [24]:
X_train, X_vaild, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_vaild.shape, y_train.shape, y_valid.shape

((315, 4), (105, 4), (315,), (105,))

### 의사결정나무 분류 모델 학습시키기

In [25]:
model = DecisionTreeClassifier(random_state=42)  # 일관성있는 테스트 데이터 추출
model.fit(X_train, y_train)  # 학습하기


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [26]:
pred = model.predict(X_vaild) # 예측값
pred

array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])

In [27]:
from sklearn.metrics import accuracy_score # 정확도

accuracy_score(y_valid, pred) # 정답과 예측을 비교해서 얼마나 정확한가?

0.5333333333333333

### 나무 깊이를 바꿔가며 훈련/검증 점수를 비교 --> 과적합을 관찰하기 위해서..

In [30]:
# 깊이를 2, 4, 제한없음(None) 세 가지로 바꿔가며 성능 비교
for depth in [2, 4, None]:
    model = make_pipeline(
        SimpleImputer(strategy='median'), # 결측치 패턴을 중앙값으로 채운다.
        DecisionTreeClassifier(max_depth=depth, random_state=42)
    )
    model.fit(X_train, y_train)  #학습
    pred = model.predict(X_vaild) #예측
    print(f'{depth} -> {accuracy_score(y_valid, pred):.4f}') #정확도 확인(소수 네째자리까지)

2 -> 0.6952
4 -> 0.6476
None -> 0.5333


- 깊이가 없음(끝까지 내려가는 것)과 깊이 2가 성능이 서로 비슷하다면 깊이가 얕은 모델을 우선한다.

### 추가) 깊이가 2인 얖은 나무모델을 직접 뜯어보기

In [34]:
shallow = make_pipeline(
    SimpleImputer(strategy='median'),
    DecisionTreeClassifier(max_depth=2, random_state=42)
)
shallow.fit(X_train, y_train)
# named_steps : 파이프라인 안에서 이름으로 특정 단계(모델)를 꺼내오는 방법
tree_model = shallow.named_steps['decisiontreeclassifier']

print(export_text(tree_model, feature_names=features))

|--- complaints <= 1.50
|   |--- contract_months <= 12.50
|   |   |--- class: 0
|   |--- contract_months >  12.50
|   |   |--- class: 0
|--- complaints >  1.50
|   |--- monthly_fee <= 32.75
|   |   |--- class: 0
|   |--- monthly_fee >  32.75
|   |   |--- class: 1

